# ETL — nguồn PNG một trang/file (KNTT, 801 trang) · cập nhật 2026-08-21

Notebook dựng lại DB cho hệ RAG SGK KHTN. **Nguồn đã đổi hoàn toàn:** không còn
PDF, `datasources/` là **4 thư mục PNG, một file mỗi trang**
(`SGK_KHTN_{6,7,8,9}_KNTT/page_001.png …`, tổng **801 trang**).

**Ba thứ khác bản notebook cũ — đọc trước khi chạy:**

1. **BƯỚC MỚI BẮT BUỘC: `--build-manifests` phải chạy TRƯỚC `--text-only`/`--etl`.**
   Đường text không đoán số trang in nữa (bản cũ fallback `index + 1` — lệch đúng
   1 trang); nó lấy số trang từ `BookManifest`, không có manifest thì **dừng**.
2. **`RAG_MANIFEST_DIR`** (biến mới): manifest đi theo **repo**, index nặng nằm ở
   **Drive**. Không đặt thì manifest bị tìm trong Drive và bạn phải copy tay.
3. **Checkpoint giờ khoá theo hash TỪNG TRANG + version.** Tải bù/sửa 1 trang thì
   chỉ trang đó chạy lại. Bump `TEXT_EXTRACTION_VERSION` / `IMAGE_EXTRACTION_VERSION`
   là cách DUY NHẤT ép làm lại toàn bộ một phía.

**Trạng thái thật của hai đường (đo được, không phải dự đoán):**

| | trạng thái | ghi chú |
|---|---|---|
| `--text-only` | **dùng được** | định danh trang 793/793 = 100,0%; ~1,6 s/trang → ~21 phút/801 trang trên 1 luồng CPU |
| `--image-only` | **chạy được, output chưa tin được** | 3/4 trang QA sai nhãn/khung hình (milestone M3 chưa xong) — xem mục 8 |

Giữ như cũ: text embedding **`BAAI/bge-m3`** + cross-encoder
**`BAAI/bge-reranker-v2-m3`**; DB đổ ra Google Drive qua `RAG_DATABASE_DIR`; secret
lấy từ **Colab Secrets**.

**Colab free có chạy được không?** Được, cho `--text-only`:

- **OCR chạy trên CPU** (tesseract, đơn luồng) — GPU không giúp gì cho phần này.
  ~1,6 s/trang → **~21 phút cho 801 trang**; free tier CPU chậm hơn thì tính ~30–45 phút.
- GPU chỉ dùng để **embedding bge-m3**; T4 của free tier là quá đủ (bge-m3 ~2 GB).
- Checkpoint resume theo TỪNG TRANG, nên Colab ngắt giữa đường gần như không mất gì
  — chạy lại đúng lệnh đó.
- Điều kiện: dùng `--profile text-etl` ở bước 4 (khỏi tải 15 GB model), và đừng
  chạy `--build-manifests` nếu manifest đã có trong repo.

**Pro/L4 + High-RAM đáng tiền khi:** chạy phía ảnh (Vintern-1B caption), serve
Qwen-3B, hoặc muốn session dài không bị ngắt. Với ETL text thuần thì không cần.

> ⚠️ **Bảo mật — làm ngay:** notebook này từng hardcode `HF_TOKEN` trong ô mã
> (đã bị lộ vào git history). **Revoke token HF cũ + GitHub PAT cũ**, tạo token
> mới, lưu vào Colab Secrets (biểu tượng 🔑 bên trái) tên `HF_TOKEN`. Ô mã dưới
> đây giờ đọc từ Secrets, không hardcode.

## 1. Clone repo

In [ ]:
# Pipeline nguồn PNG đã merge vào master.
!git clone -b master https://github.com/lcdkhoa/project-bio-rag.git

In [ ]:
%cd project-bio-rag
!git log --oneline -3

## 2. Cài dependencies + Tesseract (vie)

`poppler-utils` chỉ còn cần cho đường upload PDF legacy (`/api/etl`) — nguồn PNG
không dùng. Cài luôn cho chắc, nhẹ.

In [ ]:
!pip install -q -r requirements.txt
!apt-get -qq update
!apt-get -qq install -y poppler-utils tesseract-ocr tesseract-ocr-vie
!tesseract --version | head -1 && tesseract --list-langs | grep -x vie

## 3. Secret + env cơ bản (đặt TRƯỚC khi tải model)

`HF_TOKEN` lấy từ Colab Secrets. Mở tab 🔑 (Secrets) bên trái, thêm khoá
`HF_TOKEN`, bật *Notebook access*.

In [ ]:
import os, multiprocessing
from google.colab import userdata

# HF token từ Colab Secrets — KHÔNG hardcode (bản trước hardcode và đã làm lộ token).
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

# Đa luồng khớp số CPU thực tế
n = multiprocessing.cpu_count()
os.environ["OMP_NUM_THREADS"] = str(n)
os.environ["NUMEXPR_NUM_THREADS"] = str(n)
os.environ["OPENBLAS_NUM_THREADS"] = str(n)

os.environ["USE_GPU"] = "true"
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")), "| CPU cores:", n)

## 4. Tải model về `./models` (chạy ONLINE)

Tải **cả 6 model là ~15 GB**. Chỉ tải thứ mình cần:

| profile | model | dùng cho |
|---|---|---|
| `text-etl` | bge-m3 (~2 GB) | `--text-only` |
| `image-etl` | CLIP + OWL-ViT + Vintern (~9 GB) | `--image-only` |
| `serve` | bge-m3 + reranker + Qwen-3B + CLIP (~12 GB) | `--api` |
| `all` | tất cả (~15 GB) | mặc định |

Trên **Colab free** hãy dùng `--profile text-etl` cho lượt đầu: nhanh hơn nhiều và
không ăn hết disk. `HF_HUB_OFFLINE` chưa bật ở bước này để tải được.

In [ ]:
# Lượt ETL text: chỉ cần bge-m3
!python ./src/utils/download_models.py --save_dir ./models --profile text-etl

# Khi nào cần chạy phía ảnh / serve API thì chạy thêm:
# !python ./src/utils/download_models.py --save_dir ./models --profile image-etl
# !python ./src/utils/download_models.py --save_dir ./models --profile serve

## 5. Mount Drive + trỏ DB ra Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

# Index nặng (ChromaDB + ảnh crop) -> Drive, bền qua các phiên Colab.
os.environ["RAG_DATABASE_DIR"] = "/content/drive/MyDrive/project_bio_rag/database"

# Nguồn: 4 THƯ MỤC PNG (không còn PDF). Để trong repo thì nhanh nhất; nếu bạn giữ
# trên Drive thì trỏ vào đó — cấu trúc phải là
#   <RAG_DATA_DIR>/SGK_KHTN_6_KNTT/page_001.png ...
os.environ["RAG_DATA_DIR"] = "/content/project-bio-rag/datasources"

# MỚI: manifest (bản đồ trang + spine Bài) đi theo REPO, không theo Drive.
# Không đặt biến này thì nó bị tìm trong RAG_DATABASE_DIR/manifests và bạn phải
# copy tay từ repo sang Drive.
os.environ["RAG_MANIFEST_DIR"] = "/content/project-bio-rag/database/manifests"

for key in ("RAG_DATABASE_DIR", "RAG_DATA_DIR", "RAG_MANIFEST_DIR"):
    print(f"{key} = {os.environ[key]}")

### 5b. Kiểm tra nguồn trước khi chạy bất cứ thứ gì

Kỳ vọng: **196 / 180 / 197 / 228 = 801 trang**, mỗi trang 1094×1536 (trang
`page_001.png` là 1093 — bình thường, đừng "sửa").

In [ ]:
import torch
from src.etl.page_source import discover_page_sources
import os

print("CUDA:", torch.cuda.is_available())
total = 0
for source in discover_page_sources(os.environ["RAG_DATA_DIR"]):
    numbers = source.page_numbers()
    gaps = [n for n in range(numbers[0], numbers[-1] + 1) if n not in set(numbers)]
    total += len(numbers)
    print(f"{source.name}: {len(numbers)} trang, {numbers[0]}..{numbers[-1]}, thieu: {gaps}")
print("TONG:", total, "| shape trang mau:", source.load(numbers[0]).shape)

> **(Tuỳ chọn) Build SẠCH từ đầu.** Checkpoint resume sẽ *bỏ qua* trang đã xử lý.
> Muốn dựng lại hoàn toàn, bỏ comment ô dưới (⚠️ **XOÁ toàn bộ index ở
> `RAG_DATABASE_DIR`**). Manifest KHÔNG bị xoá vì nó nằm trong repo
> (`RAG_MANIFEST_DIR`) — đó là điều mong muốn: bản đồ trang không cần dựng lại.
>
> Cách nhẹ hơn mà không xoá gì: **bump `TEXT_EXTRACTION_VERSION`** (mục 6) — mỗi
> trang sẽ được OCR lại và chunk cũ của chính trang đó bị xoá trước khi ghi mới.

In [ ]:
# import shutil, os
# shutil.rmtree(os.environ["RAG_DATABASE_DIR"], ignore_errors=True)
# os.makedirs(os.environ["RAG_DATABASE_DIR"], exist_ok=True)
# print("Đã xoá sạch:", os.environ["RAG_DATABASE_DIR"])

## 6. Env runtime — trỏ model local + bật offline + version gate

`HF_HUB_OFFLINE=1` (model đã tải ở bước 4). Hai biến version ở cuối ô là **cách
duy nhất** ép làm lại toàn bộ một phía; không đổi thì lượt chạy sau skip sạch (đó
là ý muốn, không phải lỗi).

In [ ]:
import os
base = "/content/project-bio-rag/models"

os.environ["HF_HUB_OFFLINE"] = "1"          # model đã tải ở bước 4

# --- Text: bge-m3 + reranker ---
os.environ["EMBEDDING_MODEL"] = f"{base}/bge-m3"
os.environ["RERANK_ENABLED"] = "true"
os.environ["RERANK_MODEL"]  = f"{base}/bge-reranker-v2-m3"

# --- LLM + image models ---
os.environ["LLM_MODEL"]           = f"{base}/Qwen2.5-3B-Instruct"
os.environ["CLIP_MODEL"]          = f"{base}/clip-vit-base-patch16"
os.environ["OWL_VIT_MODEL"]       = f"{base}/owlvit-base-patch32"
os.environ["IMAGE_CAPTION_MODEL"] = f"{base}/Vintern-1B-v2"
os.environ["IMAGE_CAPTION_ENABLED"] = "true"

# --- Version gate: ĐỔI GIÁ TRỊ = ép làm lại toàn bộ phía đó ---
# Để nguyên nếu chỉ muốn chạy tiếp phần còn thiếu.
os.environ["TEXT_EXTRACTION_VERSION"]  = "v1_png_region_psm"
os.environ["IMAGE_EXTRACTION_VERSION"] = "v17_png_source"
print("text ver :", os.environ["TEXT_EXTRACTION_VERSION"])
print("image ver:", os.environ["IMAGE_EXTRACTION_VERSION"])

## 7. BƯỚC 0 (BẮT BUỘC) — dựng `BookManifest` + đọc cổng G1

Manifest là **nguồn sự thật duy nhất về số trang in**. Đường text sẽ dừng với
`ManifestMissing` nếu thiếu — nó không đoán.

- Nếu `database/manifests/*.json` **đã có trong repo** → **bỏ qua ô này**.
- Chỉ chạy lại khi ảnh nguồn thay đổi (thêm/bù/sửa trang).
- Thời gian: **~20–25 phút cho 801 trang** (1 luồng CPU).

Đọc báo cáo G1 in ra cuối: mỗi quyển phải là `offset -1` và
`ocr_confirmed .../... (100.0%)`. Chữ `PASS/FAIL` cuối dòng nói về **spine Bài**,
KHÔNG phải định danh trang — sách 7 và 9 hiện `FAIL` vì spine, và điều đó **không
chặn** ETL text (`bai_so` cố tình không được ghi vào index vì spine còn sai).

In [ ]:
!python main.py --build-manifests

In [ ]:
# Xem nhanh manifest đã dựng (KHÔNG chạy lại OCR)
import glob, json, os

for path in sorted(glob.glob(os.path.join(os.environ["RAG_MANIFEST_DIR"], "*.json"))):
    m = json.load(open(path, encoding="utf-8"))
    covers = [p["page_index"] for p in m["pages"] if p["role"] == "cover"]
    unread = [p["page_index"] for p in m["pages"]
              if p["source"] != "ocr_confirmed" and p["role"] != "cover"]
    kinds = {}
    for flag in m["flags"]:
        kinds[flag["kind"]] = kinds.get(flag["kind"], 0) + 1
    print(f"{m['book_id']}: {m['n_pages']} trang | offset {m['page_offset']} | "
          f"bìa {covers} | trang có số mà KHÔNG đọc được: {unread} | Bài {len(m['bai'])}")
    print("   flags:", kinds)

## 8. ETL — TEXT (dùng được)

OCR theo **vùng layout** (không phải cả trang) → chunk → ChromaDB. Trang bìa
(`role="cover"`) bị bỏ qua ở bước chunk, **file nguồn không bị xoá**.

Resume theo TỪNG TRANG: Colab ngắt giữa đường thì chạy lại đúng lệnh này, nó chỉ
làm phần còn thiếu và **không nhân bản chunk**.

~1,6 s/trang → **~21 phút cho 801 trang** trên 1 luồng CPU (OCR không dùng GPU).

In [ ]:
!python main.py --text-only

In [ ]:
# Còn bao nhiêu trang chưa index? (biết có bị ngắt giữa đường không)
# Chạy trong subprocess để không giữ model embedding trong RAM của notebook.
import subprocess, sys, textwrap

script = textwrap.dedent("""
    import os
    from src.etl import ProcessingStatus
    from src.etl.page_source import discover_page_sources

    status = ProcessingStatus()
    for source in discover_page_sources(os.environ["RAG_DATA_DIR"]):
        print(source.name,
              "| text còn thiếu:", len(status.pages_needing_text(source)),
              "| ảnh còn thiếu:", len(status.pages_needing_images(source)))
""")
done = subprocess.run([sys.executable, "-c", script], capture_output=True, text=True)
print(done.stdout or done.stderr[-2000:])

## 9. ETL — ẢNH (chạy được, nhưng OUTPUT CHƯA TIN ĐƯỢC)

Crop figure + caption Vintern + index CLIP. `IMAGE_EXTRACTION_VERSION=v17_png_source`
vì hình học trang đã đổi (PNG 1094×1536 thay cho bản render poppler 150 DPI).

**Trạng thái đo được (QA 4 trang thật):** không crash, sinh crop, index được; và
nhãn `Hình N.M` giờ đọc được từ pill màu (13/32 trang mẫu, 17 nhãn, cả 4 quyển —
trước đây gần như 0). NHƯNG 3/4 trang QA sai: `figure_label='Em có biết'` cho một
info-box, `label='quan sát'` thay vì `Hình 21.3`, và một crop rộng gần nửa trang.
Bước gán anchor → vùng và hình học crop cần đo lại = **milestone M3, chưa xong**.

→ Nếu chạy, hãy coi output là **bản nháp** và đi qua vòng review người ở ô kế tiếp.

In [ ]:
!python main.py --image-only

In [ ]:
# Vòng review người cho metadata ảnh (ngữ nghĩa file JSON rất dễ hiểu sai —
# đọc README §6 trước khi dùng). Xoá dấu # để chạy.
# !python main.py --export-image-review database/review_images.json
# ... sửa file JSON ...
# !python main.py --apply-image-review database/review_images.json --review-user khoa

### 9b. (Thay thế) `--etl` = text + ảnh trong một lượt

Chỉ dùng khi bạn chấp nhận trạng thái của phía ảnh ở mục 9. Đường text bên trong
`--etl` **giống hệt** `--text-only`.

In [ ]:
# !python main.py --etl

## 10. (Tuỳ chọn) Eval — Recall@k / MRR

`recall_at_k.py` không cần LLM; `evaluator.py` cần `EVAL_LLM_*`.

> ⚠️ **Testset hiện tại dựng cho 12 quyển của corpus CŨ.** Phải regenerate cho 4
> quyển KNTT (`src/test/generate_testsets.py`) trước khi tin bất kỳ con số nào.

In [ ]:
# Benchmark recall nhanh (base vs rerank, + MRR) — không gọi LLM
!python src/test/recall_at_k.py

## 11. (Tuỳ chọn) Serve API + Cloudflare tunnel để demo

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# Backend
!nohup python main.py --api --port 5000 > backend.log 2>&1 &
# Tunnel công khai
!nohup cloudflared tunnel --url http://localhost:5000 > cf_backend.log 2>&1 &

In [ ]:
import time
time.sleep(8)
print('--- backend.log ---');
!tail -n 15 backend.log
print('--- public URL ---')
!grep -o 'https://[^ ]*trycloudflare.com' cf_backend.log | head -n 1

In [ ]:
# Dừng backend + tunnel + giải phóng port
!pkill -f main.py || true
!pkill -f cloudflared || true
!fuser -k 5000/tcp || true

## 12. (Tuỳ chọn) Sao lưu DB

DB đã nằm sẵn trên Drive (`RAG_DATABASE_DIR`) nên **không cần** zip/commit. Nếu muốn tải bản zip về máy:

In [ ]:
import os
src = os.environ["RAG_DATABASE_DIR"]
!zip -r -q /content/database_backup.zip "$src"
from google.colab import files
files.download('/content/database_backup.zip')